¿Qué es el "Function Calling"? 
Imagina que le preguntas a un asistente virtual: "¿Cómo está el clima en Boston?".
Por sí solo, el modelo de IA no tiene internet en tiempo real ni sensores de temperatura. Si no tuviera Function Calling, te diría: "Lo siento, no tengo acceso al clima actual".  Con Function Calling, tú le das al modelo una "herramienta" (una función en tu código). El modelo es lo suficientemente inteligente para decir: "No sé el clima, pero veo que me diste una herramienta para averiguarlo. Por favor, ejecuta tu herramienta buscando 'Boston' y dime qué sale para que yo le pueda responder al usuario".

In [ ]:
Paso 1: Instalación y Configuración
Instalciion de uv
curl -LsSf https://astral.sh/uv/install.sh | sh

Crear entorno virtual en el directorio actual
uv venv
Activacion
source .venv/bin/activate  # Si creaste con "uv venv"
Instalacion de paquetes
#uv pip install -r requirements.txt

In [10]:
import os
import json
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

In [11]:
# 1. CARGAR VARIABLES DE ENTORNO
# Esto busca tu archivo .env y carga la variable GOOGLE_API_KEY en la memoria.
load_dotenv()

True

In [12]:
# 2. DEFINIR LA HERRAMIENTA CON LANGCHAIN
# Al poner "@tool" justo encima de la función, LangChain lee automáticamente 
# las variables de la función y su descripción ("Docstring") y construye el JSON 
# complejo por ti en segundo plano.
@tool
def get_current_weather(location: str, unit: str = "fahrenheit") -> str:
    """Obtiene el clima actual en una ubicación dada."""
    
    # Esta es nuestra función "simulada" (dummy) como en tu documento original.
    weather_info = {
        "location": location,
        "temperature": "72",
        "unit": unit,
        "forecast": ["sunny", "windy"],
    }
    return json.dumps(weather_info)

In [21]:
# 3. INICIALIZAR EL MODELO GEMINI
# Instanciamos el modelo de la familia Flash. (Nota: usamos gemini-1.5-flash porque
# es la versión de producción estable más veloz actualmente equivalente en la API).
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [22]:
# 4. ENLAZAR LA HERRAMIENTA AL MODELO (BINDING)
# Aquí le decimos al modelo: "Hola Gemini, además de tu conocimiento normal, 
# tienes permiso de usar estas herramientas de esta lista".
llm_with_tools = llm.bind_tools([get_current_weather])

In [23]:
# 5. HACER LA PREGUNTA (INVOCACIÓN)
# Enviamos el mensaje tal como en el documento original.
mensaje_usuario = "What's the weather like in Boston?"
print(f"Usuario: {mensaje_usuario}")

Usuario: What's the weather like in Boston?


In [24]:
# Ejecutamos el modelo.
respuesta = llm_with_tools.invoke(mensaje_usuario)

In [25]:
# 6. VER EL RESULTADO
# El modelo no nos responderá con texto normal. Como se dio cuenta de que necesita 
# el clima, nos devolverá una petición para usar la herramienta (un "tool_call").
print("\n--- Respuesta de la IA ---")
print(f"¿Llamó a una herramienta?: {bool(respuesta.tool_calls)}")


--- Respuesta de la IA ---
¿Llamó a una herramienta?: True


In [26]:
if respuesta.tool_calls:
    llamada = respuesta.tool_calls[0]
    print(f"Nombre de la función que quiere usar: {llamada['name']}")
    print(f"Argumentos que dedujo: {llamada['args']}")

Nombre de la función que quiere usar: get_current_weather
Argumentos que dedujo: {'location': 'Boston'}
